# Inference (Single User)

Run recommendations for one user using the saved BPR model.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Workspace/Users/vminh9560@gmail.com/recommender-service")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data_test" / "dwh_mock"
BPR_PATH = PROJECT_ROOT / "models" / "bpr" / "model.npz"
LOCAL_BGE_PATH = PROJECT_ROOT / "models" / "bge-reranker"
os.environ["BPR_CHECKPOINT_PATH"] = str(BPR_PATH)
os.environ["BGE_CHECKPOINT_PATH"] = str(LOCAL_BGE_PATH)

from src.config import settings
from src.db import LocalDataClient
from src.pipeline import (
    build_interaction_table,
    load_bpr_model,
    generate_candidates,
    build_user_profiles,
    build_item_descriptions,
    rerank_candidates,
)
from src.fallback import (
    filter_cold_books,
    get_user_interaction_counts,
    classify_users,
    get_popularity_recommendations,
    build_hybrid_recommendations,
)

# Prefer local BGE checkpoint if available
if LOCAL_BGE_PATH.exists():
    try:
        object.__setattr__(settings.reranker, "model_name", str(LOCAL_BGE_PATH))
    except Exception:
        pass

target_user_id = 1

with LocalDataClient(DATA_DIR) as client:
    interaction_df = build_interaction_table(client)

    if target_user_id is not None:
        interaction_df = interaction_df[interaction_df["user_id"] == target_user_id].copy()

    if interaction_df.empty:
        recs = get_popularity_recommendations(client, top_k=settings.reranker.top_k)
    else:
        interaction_df, _ = filter_cold_books(interaction_df)
        if interaction_df.empty:
            recs = get_popularity_recommendations(client, top_k=settings.reranker.top_k)
        else:
            bpr_model = load_bpr_model(str(BPR_PATH))
            candidates_df = generate_candidates(bpr_model, interaction_df)

            interaction_counts = get_user_interaction_counts(interaction_df)
            user_groups = classify_users([target_user_id], interaction_counts)

            if target_user_id in user_groups["warm"]:
                user_profiles = build_user_profiles(client, interaction_df)
                item_descriptions = build_item_descriptions(
                    client, candidates_df["candidate_book_id"].unique().tolist()
                )
                ranked = rerank_candidates(candidates_df, user_profiles, item_descriptions)
                recs = ranked.get(target_user_id, [])
            elif target_user_id in user_groups["hybrid"]:
                popularity_recs = get_popularity_recommendations(
                    client, top_k=settings.reranker.top_k
                )
                bpr_recs = [
                    {"book_id": int(r["candidate_book_id"]), "mf_score": float(r["mf_score"])}
                    for _, r in candidates_df.iterrows()
                ]
                recs = build_hybrid_recommendations(
                    bpr_results=bpr_recs,
                    popularity_results=popularity_recs,
                    top_k=settings.reranker.top_k,
                )
            else:
                recs = get_popularity_recommendations(client, top_k=settings.reranker.top_k)

recs

In [ ]:
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

BGE_MODEL_NAME = "BAAI/bge-reranker-base"
BGE_PATH = LOCAL_BGE_PATH
if not BGE_PATH.exists() or not (BGE_PATH / "config.json").exists():
    BGE_PATH = Path("/tmp/bge-reranker")
    BGE_PATH.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(BGE_MODEL_NAME)
    tokenizer.save_pretrained(BGE_PATH)
    model.save_pretrained(BGE_PATH)

class TwoStageRecommender(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from src.config import settings
        from src.pipeline import load_bpr_model

        self.settings = settings
        self.bpr_path = context.artifacts["bpr_checkpoint"]
        self.bge_path = context.artifacts["bge_checkpoint"]
        self.data_dir = context.artifacts["data_dir"]

        os.environ["BPR_CHECKPOINT_PATH"] = self.bpr_path
        os.environ["BGE_CHECKPOINT_PATH"] = self.bge_path
        if Path(self.bge_path).exists():
            try:
                object.__setattr__(self.settings.reranker, "model_name", self.bge_path)
            except Exception:
                pass

        self.bpr_model = load_bpr_model(self.bpr_path)

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        from src.db import LocalDataClient
        from src.pipeline import (
            build_interaction_table,
            generate_candidates,
            build_user_profiles,
            build_item_descriptions,
            rerank_candidates,
        )
        from src.fallback import (
            filter_cold_books,
            get_user_interaction_counts,
            classify_users,
            get_popularity_recommendations,
            build_hybrid_recommendations,
        )

        df = pd.DataFrame(model_input)
        if "user_id" not in df.columns:
            raise ValueError("model_input must include user_id")

        results = []
        with LocalDataClient(self.data_dir) as client:
            interaction_all = build_interaction_table(client)

            for _, row in df.iterrows():
                user_id = int(row["user_id"])
                if "top_k" in df.columns and not pd.isna(row.get("top_k")):
                    top_k = int(row["top_k"])
                else:
                    top_k = self.settings.reranker.top_k

                interaction_df = interaction_all[
                    interaction_all["user_id"] == user_id
                ].copy()

                if interaction_df.empty:
                    recs = get_popularity_recommendations(client, top_k=top_k)
                else:
                    interaction_df, _ = filter_cold_books(interaction_df)
                    if interaction_df.empty:
                        recs = get_popularity_recommendations(client, top_k=top_k)
                    else:
                        candidates_df = generate_candidates(self.bpr_model, interaction_df)
                        interaction_counts = get_user_interaction_counts(interaction_df)
                        user_groups = classify_users([user_id], interaction_counts)

                        if user_id in user_groups["warm"]:
                            user_profiles = build_user_profiles(client, interaction_df)
                            item_descriptions = build_item_descriptions(
                                client,
                                candidates_df["candidate_book_id"].unique().tolist(),
                            )
                            ranked = rerank_candidates(
                                candidates_df, user_profiles, item_descriptions
                            )
                            recs = ranked.get(user_id, [])
                        elif user_id in user_groups["hybrid"]:
                            popularity_recs = get_popularity_recommendations(
                                client, top_k=top_k
                            )
                            bpr_recs = [
                                {
                                    "book_id": int(r["candidate_book_id"]),
                                    "mf_score": float(r["mf_score"]),
                                }
                                for _, r in candidates_df.iterrows()
                            ]
                            recs = build_hybrid_recommendations(
                                bpr_results=bpr_recs,
                                popularity_results=popularity_recs,
                                top_k=top_k,
                            )
                        else:
                            recs = get_popularity_recommendations(client, top_k=top_k)

                results.append(recs)

        return pd.DataFrame({"recommendations": results})

input_example = pd.DataFrame({"user_id": [1], "top_k": [5]})
output_example = pd.DataFrame({
    "recommendations": [[
        {
            "book_id": 1,
            "mf_score": 0.5,
            "semantic_score": 0.9,
            "popularity_score": None,
            "hybrid_score": None,
            "title": None,
            "rank": 1,
        }
    ]]
})
signature = infer_signature(input_example, output_example)

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run(run_name="two-stage-inference") as run:
    mlflow.pyfunc.log_model(
        artifact_path="two_stage_recommender",
        python_model=TwoStageRecommender(),
        artifacts={
            "bpr_checkpoint": str(BPR_PATH),
            "bge_checkpoint": str(BGE_PATH),
            "data_dir": str(DATA_DIR),
        },
        registered_model_name="main.default.two_stage_recommender",
        signature=signature,
        input_example=input_example,
        pip_requirements=[
            "mlflow",
            "numpy",
            "pandas",
            "torch",
            "transformers",
            "sentencepiece",
        ],
    )
    print("MLflow run:", run.info.run_id)

print("Registered model: main.default.two_stage_recommender")